# Exploring Coherence Analyses

This is the first notebook created here to explore generally how coherence analyses can be used with siesmic data to identify signals of interest. We look at these specific aspects:
1. How coherence is defined and calculated
2. How coherence can be used to identify signals of interest
3. How coherence changes when combined with compression using singular value decomposition

In the first few cells, we explore the use of scipy.signal.coherence but in later cells we write functions that implement coherence as defined by  Welch method for two signals, x and y as:

$$Coherence(x,y) = \frac{|P_{xy}|^2}{|P_{xx}||P_{yy}|}$$
where $P_{xy}$ is the cross-spectral density of x and y, and $P_{xx}$ and $P_{yy}$ are the auto-spectral densities of x and y respectively.

In general this notebook proceeds as follows:
1. Load and display some data to be used for coherence analysis
2. Explore the use of scipy.signal.coherence
3. Explore how signals may be identified using coherence
4. Explore how coherence changes when combined with compression using singular value decomposition
5. Explore validity of coherence with scipy.signal.
6. Write functions to implement coherence as defined by Welch method


In [ ]:
import os
import sys
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal as ss
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from scipy.fft import ifft

sys.path.append(os.path.join(os.path.dirname(""), os.pardir, os.pardir))
import coherence_analysis.utils.utils as f

In [ ]:
fsize = 15
ticksize = 12

### Functions for loading data and implementing compression with randomized singular value decomposition

In [ ]:
def loadBradyHShdf5(file, normalize="yes"):
    """

    Parameters
    ----------
    file : str
        path to brady hotspring h5py data file
    normalize : str, optional
        "yes" or "no". Indicates whether or not to remove laser drift and
        normalize. The default is 'yes'.

    Returns
    -------
    data : np array
        channel by samples numpy array of data
    timestamp_arr : numpy array
        array of the timestamps corresponding to the various samples in the
        data. Timestamps for brady hotspring data are with respect to the
        beginning time of the survey.

    """
    with h5py.File(file, "r") as open_file:
        dataset = open_file["das"]
        time = open_file["t"]
        data = np.array(dataset)
        timestamp_arr = np.array(time)
    data = np.transpose(data)
    if normalize == "yes":
        nSamples = np.shape(data)[1]
        # get rid of laser drift
        med = np.median(data, axis=0)
        for i in range(nSamples):
            data[:, i] = data[:, i] - med[i]

        max_of_rows = abs(data[:, :]).sum(axis=1)
        data = data / max_of_rows[:, np.newaxis]
    return data, timestamp_arr


def randomized_SVD_comp_decomp(data, compression_factor):
    """
    Compress data with randomized SVD by compression factor and return
    reconstructed data.

    Parameters
    ----------
    data : 2-dimensional numpy array
        Data to be compressed.
    compression_factor : int/float
        Compression factor.

    Returns
    -------
    recon : 2-dimensional numpy array
        Reconstructed data after compression.
    compression_factor : float/int
        Same as input compression_factor

    """
    from sklearn.utils.extmath import randomized_svd

    rows, columns = data.shape
    approxRank = int(
        (rows * columns) / (compression_factor * (rows + columns))
    )
    # calculate randomized SVD and reconstruct
    U, S, Vt = randomized_svd(data, n_components=approxRank)

    plt.plot(S)
    recon = U @ np.diag(S) @ Vt

    return recon, compression_factor

### Load data

The data used in this notebook can be downloaded via the [AWS S3 Explorer for the Open Energy Data Initiative](https://data.openei.org/s3_viewer?bucket=nrel-pds-porotomo&prefix=DAS%2FH5%2FDASH%2F). The data is stored in the `nrel-pds-porotomo` bucket and the `DAS/H5/DASH/` prefix. The specific file used in this notebook are `PoroTomo_iDAS16043_160314083848.h5` and `PoroTomo_iDAS16043_160314083918.h5`.

In [ ]:
# Modify the file paths as needed
data_dir = Path("D:/CSM/Mines_Research/Test_data/Brady_Hotspring")
file = data_dir / "PoroTomo_iDAS16043_160314083848.h5"
data, _ = f.load_brady_hdf5(file, normalize="no")

file = data_dir / "PoroTomo_iDAS16043_160314083918.h5"
data2, _ = f.load_brady_hdf5(file, normalize="no")

# signalToUse=np.append(data[:,24976:],data2[:,:10000],axis=1)
# signalToUse=np.append(data[:,24976:],data2,axis=1)

samples_per_sec = 1000

### Display part of data containing signals of interest

This data contains a cataloged microseismic event. We will use this data to explore coherence analysis techniques.

In [ ]:
pdata = data[:, 10000:25000]
start_ch = 3100
nchannels = 5100
pdata = pdata[start_ch : nchannels + start_ch]
fig2 = plt.figure()
img2 = plt.imshow(
    pdata,
    cmap="RdBu",
    vmin=-np.percentile(np.absolute(pdata), 90),
    vmax=np.percentile(np.absolute(pdata), 90),
    aspect="auto",
    interpolation="none",
    extent=(
        0,
        len(pdata[0]) / samples_per_sec,
        start_ch,
        start_ch + nchannels,
    ),
)
# extent=(mdates.date2num(np.datetime64(props['GPSTimeStamp'])),mdates.date2num(np.datetime64(props['GPSTimeStamp'])+np.timedelta64(60,'s')), distances[0],distances[-1]))
plt.xlabel("Time (seconds)", fontsize=fsize)
plt.ylabel("Sensors", fontsize=fsize)
plt.title("Signal", fontsize=fsize)
plt.xticks(fontsize=fsize)
plt.yticks(fontsize=fsize)
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=fsize)

In [ ]:
time_data = np.linspace(0, len(pdata[0]) / samples_per_sec, len(pdata[0]))
plt.plot(time_data, pdata[1000], "k", linewidth=2)
plt.plot(time_data, pdata[900] + 0.02, "c", linewidth=2)
plt.xticks(fontsize=ticksize)
plt.yticks(fontsize=ticksize)
plt.xlabel("Time (seconds)", fontsize=fsize)
plt.ylabel("Amplitude", fontsize=fsize)
# plt.xlim(0, len(pdata[0]) / samples_per_sec)

### Define and display signal for coherence analysis.

This is a short segment of data that contains the signal of interest.

In [ ]:
pdata = np.append(data[:, 10000:], data2, axis=1)
pdata = pdata[start_ch : nchannels + start_ch]
fig2 = plt.figure()
img2 = plt.imshow(
    pdata,
    cmap="RdBu",
    vmin=-np.percentile(np.absolute(pdata), 90),
    vmax=np.percentile(np.absolute(pdata), 90),
    aspect="auto",
    interpolation="none",
    extent=(
        0,
        len(pdata[0]) / samples_per_sec,
        start_ch,
        start_ch + nchannels,
    ),
)
# extent=(mdates.date2num(np.datetime64(props['GPSTimeStamp'])),mdates.date2num(np.datetime64(props['GPSTimeStamp'])+np.timedelta64(60,'s')), distances[0],distances[-1]))
plt.xlabel("Time (seconds)", fontsize=fsize)
plt.ylabel("Sensors", fontsize=fsize)
plt.title("Signal", fontsize=fsize)
plt.xticks(fontsize=fsize)
plt.yticks(fontsize=fsize)
cbar = plt.colorbar()
cbar.ax.tick_params(labelsize=fsize)

### Plot coherence between 2 signals. 

Right now, we are using the `scipy.signal.coherence` function to compute the coherence between two signals. Later in this notebook and in later notebooks, we write custom functions to compute coherence.

From the plot below, we see that the coherence between the two signals is high at the frequency of the signal of interest. However, the coherence above 350 Hz is stuck at 1. This is something that we attributed later on to a lowpass filter that might have been applied to the data before we downloaded it.

In [ ]:
frequencies, coherences = ss.coherence(
    pdata[1000], pdata[1500], fs=1000
)  # , nperseg=1024)

plt.figure(figsize=(15, 5))
plt.subplot(121)
plt.plot(frequencies, coherences)
plt.xlabel("frequency [Hz]")
plt.ylabel("Coherence")
plt.subplot(122)
plt.plot(frequencies[:80], coherences[:80])
plt.xlabel("frequency [Hz]")

### Plot ifft of previously computed coherence between signals. 

At this point, we are just exploring how coherence can be used to identify signals of interest. This plot does not look particularly interesting.

In [ ]:
rt = ifft(coherences)
wt = np.zeros_like(rt)
halfpoint = int(len(rt) / 2)
wt[: halfpoint + 1] = rt[halfpoint:]
wt[halfpoint + 1 :] = rt[:halfpoint]
plt.plot(wt)

### Now we consider 200 channels and compute the coherence matrix at the various frequencies.

The coherence matrix has entries that represent the coherence between the various pairs of sensors and hence are between 0 and 1. So the entry {i,j} represents the coherence between sensors i and j. The diagonal entries are 1, and coherence matrix is symmetric. 

In [ ]:
nsensors = 200
i = 0
j = 0
coherence_cube = np.empty([nsensors, nsensors, len(coherences)])
for a in np.linspace(0, len(pdata) - 1, nsensors):
    for b in np.linspace(0, len(pdata) - 1, nsensors):
        a = int(a)
        b = int(b)
        if a <= b:
            frequencies, coherences = ss.coherence(
                pdata[a], pdata[int(b)], fs=1000
            )
            coherence_cube[i, j, :] = coherences
        else:
            coherence_cube[i, j, :] = coherence_cube[j, i, :]
        j += 1
    i += 1
    j = 0

### Show coherence matrix at a particular frequency

In [ ]:
plt.imshow(coherence_cube[:, :, 5])

### Make an animation of the coherence matrix at various frequencies.

This is to show how coherence changes with frequency, and whether there are any frequencies that stand out.

In [ ]:
# Create two 3D arrays (example data)
num_frames = len(coherences)  # 30
data1 = coherence_cube[:]  # Replace with your 3D data array 1

# Create a function to update the animations
fig = plt.figure(figsize=(6, 6))


def update(frame):
    fig.clear()

    # Update the first animation
    ax = fig.add_subplot(111)
    ax.imshow(
        data1[:, :, frame],
        cmap=plt.cm.viridis,
        animated=True,
        extent=[0, nsensors, 0, nsensors],
    )  # ,  vmin=-np.percentile(abs(data1[frame]),90), vmax=np.percentile(abs(data1[frame]),90))
    # ax.imshow(data1[frame], cmap=plt.cm.viridis, animated=True,  vmin=-epsilon, vmax=epsilon)
    ax.set_title("Frequency = " + str(frequencies[frame]), size=fsize)

    plt.xticks(fontsize=ticksize)
    plt.yticks(fontsize=ticksize)
    plt.tight_layout()


# Create the animations
fig = plt.figure(figsize=(6, 6))
ani = FuncAnimation(fig, update, frames=num_frames, repeat=False)

# Display the animations side by side
display(HTML(ani.to_jshtml()))

# Alternatively, you can save the animations as video files
# ani.save('Coherence_mat.gif', writer='imagemagick')

### Plot eigenvalue decay at a particular frequency

This is to explore how different patterns at the various frequencies show up in the eigenvalue decay. In the end, the eigenvalue decay of the coherence matrix is what we will use to identify signals of interest.

In [ ]:
frame = 2
eigenvals, eigenvecs = np.linalg.eig(coherence_cube[:, :, frame])
eigenvals = np.sort(eigenvals)[::-1]
plt.plot(
    eigenvals[:] / np.sum(eigenvals[:]),
    "-x",
    label=f"frequency {frequencies[frame]} Hz",
)

frame = 6
eigenvals, eigenvecs = np.linalg.eig(coherence_cube[:, :, frame])
eigenvals = np.sort(eigenvals)[::-1]
plt.plot(
    eigenvals[:] / np.sum(eigenvals[:]),
    "-s",
    label=f"frequency {frequencies[frame]} Hz",
)
plt.xlabel("Eigenvalue index")
plt.ylabel("Eigenvalue")
plt.title("Eigenvalues of the coherence matrix")
plt.legend(fontsize=fsize)
# eigenvals[0]

### Plot the ratio of the first eigenvalue to the sum of all the eigenvalues of the coherence matrix at various frequencies

Here, we are using the first eigenvalue as an indication of a signal of interest and we want to see how it changes with frequency.

In [ ]:
num_frames = len(coherences)
eig_ratios = np.empty(num_frames)
for a in range(num_frames):
    eigenvals, _ = np.linalg.eig(coherence_cube[:, :, a])
    eigenvals = np.sort(eigenvals)[::-1]
    eig_ratios[a] = eigenvals[0] / np.sum(eigenvals)
plt.plot(frequencies[:80], eig_ratios[:80], "-x")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Eigenvalue ratio")
plt.title("Ratio of the first two eigenvalues of the coherence matrix")

### Viewing the eigenvalue decay at various frequencies

We can see that the value of the first eigenvalue seems to be high at various frequencies that may contain of the signal of interest. This could be an indication that the first eigenvalue can be used to identify signals of interest.

In [ ]:
num_frames = len(coherences)  # 30
data1 = coherence_cube[:]  # Replace with your 3D data array 1

# Create a function to update the animations
fig = plt.figure(figsize=(6, 6))


def update(frame):
    fig.clear()

    # Update the first animation
    ax = fig.add_subplot(111)
    eigenvals, _ = np.linalg.eig(coherence_cube[:, :, frame])
    eigenvals = np.sort(eigenvals)[::-1]
    eigenvals = eigenvals / np.sum(eigenvals)
    plt.plot(eigenvals, "-o")
    plt.ylim([0, 1])
    ax.set_title("Frequency = " + str(frequencies[frame]), size=16)

    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)

    plt.tight_layout()


# Create the animations
fig = plt.figure(figsize=(6, 6))
ani = FuncAnimation(fig, update, frames=num_frames, repeat=False)

# Display the animations side by side
display(HTML(ani.to_jshtml()))

# Alternatively, you can save the animations as video files
# ani.save('Coherence_mat.gif', writer='imagemagick')

# Other analyses considered

In the remaining cells, we consider other ways of computing the spectral density. We also consider the use of singular value decomposition to compress the data and then compute coherence. We want to see how coherence changes when the data is compressed.

Explore other ways of computing the spectral density using scipy.

In [ ]:
# plt.plot(pdata[1])
# plt.plot(ss.welch(pdata[50]))
f, Pxy = ss.welch(pdata[50], fs=1000)
plt.plot(f, Pxy)
f, Pxx = ss.csd(pdata[50], pdata[50], fs=1000)
plt.plot(f, Pxx, "--")
print(len(f))

Check out the spectrogram and how it connects to the data at a particular channel.

In [ ]:
f, t, Sxx = ss.spectrogram(
    data[5253], fs=1000
)  # , nperseg=1000, noverlap=500, nfft=1000)
plt.subplot(211)
plt.pcolormesh(t, f, Sxx, shading="gouraud", cmap="turbo")
plt.ylabel("Frequency [Hz]")
plt.xlabel("Time [sec]")
plt.subplot(212)
plt.plot(data[5253])

### Select parts of data, do a randomized SVD and plot singular values

This is just to see decay of the singular values and how much compression is possible for our data using randomized SVD.

In [ ]:
# pdata=data[:,5000:25000]
# pdata = pdata[start_ch:nchannels+start_ch]
pdata, _ = randomized_SVD_comp_decomp(pdata, 1)

### Test out how the coherence changes when data is compressed using randomized SVD

This did not seem very promising so we did not explore it further.

In [ ]:
from sklearn.utils.extmath import randomized_svd

# compression_factors = [1,5,10,15,20,25,50,100]
compression_factors = [1, 50, 100]
pdata = data[:, 10000:25000]
pdata = pdata[start_ch : nchannels + start_ch]
rows, columns = pdata.shape
nsensors = 200
evs = np.array([range(nsensors)])


approxRank = int(
    (rows * columns) / (compression_factors[0] * (rows + columns))
)
# calculate randomized SVD and reconstruct
U, S, Vt = randomized_svd(pdata, n_components=approxRank)

for compression_factor in compression_factors:
    approxRank = int(
        (rows * columns) / (compression_factor * (rows + columns))
    )
    recon = U[:, :approxRank] @ np.diag(S[:approxRank]) @ Vt[:approxRank, :]

    i = 0
    j = 0
    cube = np.empty([nsensors, nsensors, len(coherences)])
    for a in np.linspace(0, len(recon) - 1, nsensors):
        for b in np.linspace(0, len(recon) - 1, nsensors):
            a = int(a)
            b = int(b)
            # f, C = ss.coherence(recon[a], recon[int(b)], fs=1000)
            # cube[i,j,:] = C
            if a <= b:
                f, C = ss.coherence(recon[a], recon[b], fs=1000)
                cube[i, j, :] = C
            else:
                cube[i, j, :] = cube[j, i, :]
            j += 1
        i += 1
        j = 0
    eigenvals, eigenvecs = np.linalg.eig(cube[:, :, 2])
    eigenvals = np.sort(eigenvals)[::-1]
    # plt.plot(eigenvals, '-x', label="Comp rate: "+ str(compression_factor))
    # evs = np.append(evs, eigenvals[np.newaxis,:], axis=0)
    num_frames = len(C)
    eig_ratios = np.empty(num_frames)
    for d in range(num_frames):
        eigenvals, _ = np.linalg.eig(cube[:, :, d])
        eigenvals = np.sort(eigenvals)[::-1]
        eig_ratios[d] = eigenvals[0] / eigenvals[1]
    plt.plot(
        f, eig_ratios, "-x", label="Comp rate: " + str(compression_factor)
    )
plt.legend()

The next cell is to investigate some intuition that the compression error from SVD is proportional to the product of the first singular value with each of the other singular values. 

In [ ]:
# np.append(eigenvals[np.newaxis,:], eigenvals[np.newaxis,:], axis=0).shape
# evs = np.array([range(200)])
# evs
approxRank = int(
    (rows * columns) / (compression_factors[0] * (rows + columns))
)
# calculate randomized SVD and reconstruct
# U, S, Vt = randomized_svd(data, n_components=approxRank)
plt.plot(S, "-o")
plt.plot(S * S[0], "-s")

In [ ]:
plt.plot(pdata[50])

### Define new functions for computing coherence matrix

In [ ]:
def windowed_spectra(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    win_start = 0
    window_samples = subwindow_len // sample_interval
    total_samples = data.shape[-1]
    intervals = np.arange(
        window_samples, total_samples + 1, window_samples, dtype=int
    )  # break time series into windowed intervals

    win_end = intervals[0]

    absolute_spectra = np.fft.rfft(data[:, win_start:win_end])
    win_spectra = absolute_spectra[np.newaxis]

    # win_start = intervals[0]
    # win_start = win_end - overlap
    # for win_end in intervals[1:]:  # for each interval, calculate and record its spectrum
    while win_end < total_samples:
        win_start = win_end - overlap
        win_end = win_start + window_samples
        absolute_spectra = np.fft.rfft(data[:, win_start:win_end])
        win_spectra = np.append(
            win_spectra, absolute_spectra[np.newaxis], axis=0
        )
        # win_start = win_end

    frequencies = np.fft.rfftfreq(window_samples, sample_interval)

    return win_spectra, frequencies


def welch_coherence(
    data: np.array, subwindow_len: int, overlap, freq=None, sample_interval=1
):
    win_spectra, frequencies = windowed_spectra(
        data, subwindow_len, overlap, freq, sample_interval
    )

    nomalizer = np.sum(np.absolute(win_spectra) ** 2, axis=0)
    nomalizer = np.tile(nomalizer, (nomalizer.shape[0], 1, 1)) * np.tile(
        nomalizer, (nomalizer.shape[0], 1, 1)
    )
    nomalizer = nomalizer * nomalizer.transpose((1, 0, 2))
    nomalizer = nomalizer.transpose(2, 1, 0)

    welch_numerator = np.matmul(
        win_spectra.transpose(2, 1, 0),
        np.conjugate(win_spectra.transpose(2, 0, 1)),
    )
    welch_numerator = np.absolute(welch_numerator) ** 2
    coherence = np.multiply(welch_numerator, 1 / nomalizer)

    return coherence, frequencies

### Implement computation on a random signal

In [ ]:
a = np.random.rand(5, 5000)
# win_absolute_spectra, frequencies = windowed_absolute_spectra(a, 500,0)
win_spectra, frequencies = windowed_spectra(a, 500, 0)
nomalizer = np.sum(np.absolute(win_spectra) ** 2, axis=0)
nomalizer = np.tile(nomalizer, (nomalizer.shape[0], 1, 1))
nomalizer = nomalizer * nomalizer.transpose((1, 0, 2))
nomalizer = nomalizer.transpose(2, 1, 0)

welch_numerator = np.matmul(
    win_spectra.transpose(2, 1, 0),
    np.conjugate(win_spectra.transpose(2, 0, 1)),
)
welch_numerator = np.absolute(welch_numerator) ** 2
coherence = np.multiply(welch_numerator, 1 / nomalizer)

# nomalizer.shape
# coherence, frequencies = welch_coherence(a, 500,0)
# coherence[1]

### Check that the maximum value of the coherence matrix is 1.

In [ ]:
np.max(coherence)